# Day 12: Used Car Data Preprocessing

This notebook preprocesses the provided used-car resale dataset for a regression model. It inspects data quality, identifies outliers with the IQR rule, encodes nominal and ordinal categorical variables, scales numeric features, separates the target, and fits transformations only on the training data to prevent leakage.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

DATA_PATH = Path('Day12_Used_Car_Preprocessing_Dataset.csv')
PROCESSED_PATH = Path('used_car_preprocessed.csv')
TARGET = 'Resale_Price_Lakh'

if not DATA_PATH.exists():
    raise FileNotFoundError(f'Dataset not found: {DATA_PATH.resolve()}')

df = pd.read_csv(DATA_PATH)
print('Dataset shape:', df.shape)
display(df.head())
print('\nData types:\n', df.dtypes)
print('\nMissing values:\n', df.isnull().sum())
print('\nDuplicate rows:', int(df.duplicated().sum()))
print('\nCategorical values:')
for column in ['Brand', 'Fuel_Type', 'Transmission', 'City', 'Seller_Type', 'Condition']:
    print(f'{column}: {sorted(df[column].dropna().astype(str).unique())}')

assert TARGET in df.columns
assert df[TARGET].notna().all()

# Separate the identifier, features, and regression target before splitting.
identifiers = df['Car_ID'].copy()
X = df.drop(columns=['Car_ID', TARGET])
y = df[TARGET].copy()

X_train, X_test, y_train, y_test, id_train, id_test = train_test_split(
    X, y, identifiers, test_size=0.2, random_state=42
)

# Detect outliers using IQR bounds calculated from training data only.
outlier_columns = [
    'Year', 'Mileage_Km', 'Engine_CC', 'Power_BHP',
    'Previous_Owners', 'Accidents_Reported', 'Service_Score'
]
q1 = X_train[outlier_columns].quantile(0.25)
q3 = X_train[outlier_columns].quantile(0.75)
iqr = q3 - q1
lower_bounds = q1 - 1.5 * iqr
upper_bounds = q3 + 1.5 * iqr
outlier_mask = ((X_train[outlier_columns] < lower_bounds) | (X_train[outlier_columns] > upper_bounds)).any(axis=1)
print('\nTraining rows identified as IQR outliers:', int(outlier_mask.sum()))
print('Outliers are removed from training only; test rows remain untouched for honest evaluation.')

X_train = X_train.loc[~outlier_mask].copy()
y_train = y_train.loc[X_train.index]
id_train = id_train.loc[X_train.index]

condition_order = {'Poor': 1, 'Fair': 2, 'Good': 3, 'Very Good': 4, 'Excellent': 5}
for frame in [X_train, X_test]:
    frame['Condition_Ordinal'] = frame['Condition'].map(condition_order)
    frame.drop(columns=['Condition'], inplace=True)

numeric_features = [
    'Year', 'Mileage_Km', 'Engine_CC', 'Power_BHP',
    'Previous_Owners', 'Accidents_Reported', 'Service_Score', 'Condition_Ordinal'
]
nominal_features = ['Brand', 'Fuel_Type', 'Transmission', 'City', 'Seller_Type']

numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
nominal_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])
preprocessor = ColumnTransformer([
    ('numeric', numeric_pipeline, numeric_features),
    ('nominal', nominal_pipeline, nominal_features)
])

# Fit the imputer, encoder, and scaler on training data only.
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)
feature_names = preprocessor.get_feature_names_out()

X_train_processed = pd.DataFrame(X_train_processed, columns=feature_names, index=X_train.index)
X_test_processed = pd.DataFrame(X_test_processed, columns=feature_names, index=X_test.index)

train_output = X_train_processed.copy()
train_output[TARGET] = y_train
train_output['Dataset_Split'] = 'train'
test_output = X_test_processed.copy()
test_output[TARGET] = y_test
test_output['Dataset_Split'] = 'test'
processed_df = pd.concat([train_output, test_output]).sort_index()
processed_df.to_csv(PROCESSED_PATH, index=False)

print('\nTraining shape after outlier removal:', X_train_processed.shape)
print('Testing shape:', X_test_processed.shape)
print('Processed feature count:', len(feature_names))
print('Saved:', PROCESSED_PATH)
display(processed_df.head())

Dataset shape: (320, 15)


,Car_ID,Brand,Year,Mileage_Km,Engine_CC,Power_BHP,Fuel_Type,Transmission,City,Seller_Type,Condition,Previous_Owners,Accidents_Reported,Service_Score,Resale_Price_Lakh
0,CAR0001,Skoda,2021,69708,1152,128.8,Diesel,Manual,Lucknow,Individual,Good,1,0,72,6.38
1,CAR0002,Toyota,2020,88881,903,146.5,Diesel,Automatic,Chandigarh,Individual,Good,1,0,87,4.83
2,CAR0003,Volkswagen,2021,43646,1446,185.9,Diesel,Automatic,Hyderabad,Individual,Very Good,2,0,90,7.30
3,CAR0004,Tata,2019,70847,2069,148.8,Petrol,Manual,Lucknow,Individual,Excellent,3,0,66,3.82
4,CAR0005,Tata,2016,101228,1657,206.0,Petrol,Automatic,Ahmedabad,Dealer,Very Good,2,0,84,1.93



Data types:
 Car_ID                    str
Brand                     str
Year                    int64
Mileage_Km              int64
Engine_CC               int64
Power_BHP             float64
Fuel_Type                 str
Transmission              str
City                      str
Seller_Type               str
Condition                 str
Previous_Owners         int64
Accidents_Reported      int64
Service_Score           int64
Resale_Price_Lakh     float64
dtype: object

Missing values:
 Car_ID                0
Brand                 0
Year                  0
Mileage_Km            0
Engine_CC             0
Power_BHP             0
Fuel_Type             0
Transmission          0
City                  0
Seller_Type           0
Condition             0
Previous_Owners       0
Accidents_Reported    0
Service_Score         0
Resale_Price_Lakh     0
dtype: int64

Duplicate rows: 0

Categorical values:
Brand: ['Honda', 'Hyundai', 'Kia', 'Mahindra', 'Maruti', 'Renault', 'Skoda', 'Tata', 'Toyot

,numeric__Year,numeric__Mileage_Km,numeric__Engine_CC,numeric__Power_BHP,numeric__Previous_Owners,numeric__Accidents_Reported,numeric__Service_Score,numeric__Condition_Ordinal,nominal__Brand_Honda,nominal__Brand_Hyundai,...,nominal__City_Jaipur,nominal__City_Kochi,nominal__City_Lucknow,nominal__City_Mumbai,nominal__City_Pune,nominal__Seller_Type_Certified Dealer,nominal__Seller_Type_Dealer,nominal__Seller_Type_Individual,Resale_Price_Lakh,Dataset_Split
0,0.373945,-0.076269,-0.420807,-0.698221,-0.719667,0.0,-0.433476,-0.315994,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,6.38,train
1,0.075392,0.481294,-1.034794,-0.095352,-0.719667,0.0,0.818882,-0.315994,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,4.83,train
2,0.373945,-0.834169,0.304142,1.246627,0.677334,0.0,1.069354,0.709692,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,7.30,train
3,-0.223161,-0.043146,1.840342,-0.017013,2.074335,0.0,-0.934420,1.735379,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,3.82,test
4,-1.118821,0.840353,0.824428,1.931241,0.677334,0.0,0.568411,0.709692,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.93,train


## Verification

The output is checked after transformation. Numeric features are standardized using statistics learned from the training subset only, nominal categories are one-hot encoded, and the ordinal condition is mapped from Poor to Excellent.

In [3]:
numeric_output_columns = [name for name in feature_names if name.startswith('numeric__')]
training_numeric_std = X_train_processed[numeric_output_columns].std(ddof=0)
non_constant_numeric = training_numeric_std[training_numeric_std > 0].index
constant_numeric = training_numeric_std[training_numeric_std == 0].index.tolist()

assert PROCESSED_PATH.exists()
assert processed_df.isnull().sum().sum() == 0
assert np.isfinite(processed_df.drop(columns=['Dataset_Split']).select_dtypes(include='number')).all().all()
assert set(processed_df['Dataset_Split']) == {'train', 'test'}
assert len(X_train_processed) + len(X_test_processed) == len(processed_df)
assert np.allclose(X_train_processed[numeric_output_columns].mean(), 0, atol=1e-10)
assert np.allclose(training_numeric_std[non_constant_numeric], 1, atol=1e-10)

print('Processed dataset shape:', processed_df.shape)
print('Processed missing cells:', int(processed_df.isnull().sum().sum()))
print('Training rows:', len(X_train_processed))
print('Testing rows:', len(X_test_processed))
print('Numeric training means approximately zero:', np.allclose(X_train_processed[numeric_output_columns].mean(), 0, atol=1e-10))
print('Non-constant numeric training standard deviations approximately one:', np.allclose(training_numeric_std[non_constant_numeric], 1, atol=1e-10))
print('Constant numeric training features:', constant_numeric)
print('Verification passed: preprocessing is complete and transformations were fitted on training data only.')

Processed dataset shape: (262, 39)
Processed missing cells: 0
Training rows: 198
Testing rows: 64
Numeric training means approximately zero: True
Non-constant numeric training standard deviations approximately one: True
Constant numeric training features: ['numeric__Accidents_Reported']
Verification passed: preprocessing is complete and transformations were fitted on training data only.
